# **Proyecto 04: Sistema de Sentimiento en Reseñas de Usuario**

---
## **Parte 1: Carga del data set en la librería de Python**

En este proyecto, en lugar de cargar documentos de tipo '.csv' o de tipo '.txt', vamos a utilizar una librería de Python que ya tiene el dataset integrado. Para ello, importaremos la librería [**gensim**](https://radimrehurek.com/gensim/), y llamaremos a la función [**load_word2vec_format**](https://radimrehurek.com/gensim/models/word2vec.html). Con esta función cargaremos un modelo preconstruido de Word2Vec, propocionado por Google, que ha sido entrenado sobre billones de páginas de texto y que ha producido como salida un vector de dimensión 300 para todas las diferentes palabras. Una vez cargado el modelo, se puede ver el vector de 300 valores que representa una palabra en concreto, como por ejemplo <span style="color:red"> **cat**</span> o <span style="color:red"> **dog**</span>.

Podemos incluso mirar el coeficiente de similaridad entre dos palabras distintas, utilizando la función [**similarity**](https://tedboy.github.io/nlps/generated/generated/gensim.models.Word2Vec.similarity.html).

Para entrenar el modelo de Word2Vec, cargamos las clases <span style="color:blue"> **Doc2Vec**</span> y <span style="color:blue"> **TaggedDocument**</span>. Vamos a utilizar la primera porque queremos determinar un vector para cada documento, no necesariamente para cada palabra en el documento, ya que nuestros documentos son reseñas y nosotros queremos ver si esas reseñas son positivas o negativas, lo que significa comprobar si son similares a reseñas positivas o similares a reseñas negativas. <span style="color:blue"> **Doc2Vec**</span> lo proporciona la librería <span style="color:blue"> **gensim**</span> y también contiene la clase <span style="color:blue"> **TaggedDocument**</span>, que permite usar "estas son las palabras en el documento, y Doc2Vec es el modelo".

Creamos a continuación una función de utilidad que tomará una frase o un párrafo entero y le quitará las mayúsculas y eliminará las etiquetas HTML, apóstrofes, puntuación, espacios y espacios repetidos, y por último devolverá el párrafo separado por palabras (utilizad la función [**split**](https://www.w3schools.com/python/ref_string_split.asp) para esto último.

---

## **Parte 2: Carga de los datos**

Ahora vamos a cargar los datos sobre las reseñas de películas. En esta ocasión vamos a utilizar reseñas que no están catalogadas ni como positivas ni como negativas para entrenar el modelo; vamos a efectuar lo que se conoce como un unsupervised learning. Para cargar los datos, necesitamos hacer un bucle sobre el directorio donde se encuentran los archivos, así como otro bucle sobre los archivos de dicho directorio, tal y como se muestra a continuación

In [11]:

import re
import os
unsup_sentences = []

# fuente: http://ai.stanford.edu/~amaas/data/sentiment/, datos de IMDB
for dirname in ["train/pos", "train/neg", "train/unsup", "test/pos", "test/neg"]:
    for fname in sorted(os.listdir("aclImdb/" + dirname)):
        if fname[-4:] == '.txt':
            with open("aclImdb/" + dirname + "/" + fname, encoding='UTF-8') as f:
                sent = f.read()
                words = extract_words(sent)
                unsup_sentences.append(TaggedDocument(words, [dirname + "/" + fname]))

# fuente: http://www.cs.cornell.edu/people/pabo/movie-review-data/
for dirname in ["review_polarity/txt_sentoken/pos", "review_polarity/txt_sentoken/neg"]:
    for fname in sorted(os.listdir(dirname)):
        if fname[-4:] == '.txt':
            with open(dirname + "/" + fname, encoding='UTF-8') as f:
                for i, sent in enumerate(f):
                    words = extract_words(sent)
                    unsup_sentences.append(TaggedDocument(words, ["%s/%s-%d" % (dirname, fname, i)]))
                
# fuente: https://nlp.stanford.edu/sentiment/, datos de Rotten Tomatoes
with open("stanfordSentimentTreebank/original_rt_snippets.txt", encoding='UTF-8') as f:
    for i, line in enumerate(f):
        words = extract_words(sent)
        unsup_sentences.append(TaggedDocument(words, ["rt-%d" % i]))

---

## **Parte 3: Análisis del model (entrenamiento)**

El entrenamiento sin supervisión lo realizamos primero cambiando de orden los datos de entrenamiento (con la función <span style="color:blue"> **shuffle**</span>) y entrenamos llamando al modelo <span style="color:blue"> **Doc2Vec**</span> (utilizad los parámetros dm=0, hs=1 y size=50). Puede tardar entre 5 y 10 minutos en efectuar este entrenamiento.

Una vez terminado el entrenamiento, eliminamos los datos de entrenamiento con la función <span style="color:blue"> **delete_temporary_training_data**</span> y guardamos el modelo en un archivo ".d2v".

Una vez el modelo ha sido entrenado, podemos inferir un vector. Probad a mandar una reseña a la función infer_vector, por ejemplo "This place is not worth your time, let alone Vegas.". Podéis probar con otras como "Service sucks" o "Highly recommended". Si queréis comparar entre dos reseñas, podéis llamar a similarity, aunque
podemos probar con [**cosine_similarity**](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)

El modelo ha aprendido como se usan las palabras juntas en la misma reseña y que un grupo de palabras pueden ir juntas con un significado, y otro grupo de palabras pueden ir juntas con otro resultado. 

A continuación cargamos el dataset para predecir (aquellas carpetas que sí que tienen etiquetas)

In [21]:
sentences = []
sentvecs = []
sentiments = []
for fname in ["yelp", "amazon_cells", "imdb"]: 
    with open("sentiment labelled sentences/%s_labelled.txt" % fname, encoding='UTF-8') as f:
        for i, line in enumerate(f):
            line_split = line.strip().split('\t')
            sentences.append(line_split[0])
            words = extract_words(line_split[0])
            sentvecs.append(model.infer_vector(words, steps=10)) # crea un vector para este documento
            sentiments.append(int(line_split[1]))
            
# baraja las frases (sentences), vectores de sentimiento (sentvecs) y sentimientos (sentiments) juntos
combined = list(zip(sentences, sentvecs, sentiments))
random.shuffle(combined)
sentences, sentvecs, sentiments = zip(*combined)

Para clasificar este modelo entrenado que hemos obtenido, vamos a utilizar un clasificador de k-vecinos cercanos, utilizando la clase [*+KNeighborsClassifier**](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) y un [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html), así como la función [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) para evaluar y verificar los resultados.